# Trump Tweet Sentiment Analysis

This notebook analyzes approximately 661,000 tweets mentioning Donald Trump using three complementary approaches:
- **NRCLex**: Lexicon-based emotion classification (10 emotion categories)
- **TextBlob**: Polarity-based sentiment classification (Positive, Negative, Neutral)
- **RoBERTa**: Transformer-based emotion detection trained on Twitter data

In [ ]:
# ── Setup & Imports ────────────────────────────────────────────────────────
!pip install -q nrclex langdetect langid swifter transformers torch

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nrclex import NRCLex
import langid
import swifter
from textblob import TextBlob
from collections import Counter
from transformers import pipeline

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/processedtrumpv3.csv"

from google.colab import drive
drive.mount("/content/drive")

data_trump = pd.read_csv(DATA_PATH, lineterminator="\n")
print(f"Dataset shape: {data_trump.shape}")
data_trump.head()

In [ ]:
# ── Language Filtering ─────────────────────────────────────────────────────
data_trump['lang'] = data_trump['tweet'].astype(str).swifter.apply(lambda x: langid.classify(x)[0])
data_trump = data_trump[data_trump['lang'] == 'en'].reset_index(drop=True)
print(f"English tweets: {data_trump.shape[0]:,}")

## 1. NRCLex Emotion Classification

NRCLex is a lexicon-based approach that classifies text into 10 emotion categories based on word associations: anger, anticipation, disgust, fear, joy, negative, positive, sadness, surprise, and trust.

In [ ]:
def nrc_dominant(text):
    """Return the dominant NRC emotion for a given text."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    emo = NRCLex(text)
    if emo.raw_emotion_scores:
        return max(emo.raw_emotion_scores, key=emo.raw_emotion_scores.get)
    return np.nan

data_trump['nrc_emotion'] = data_trump['tweet'].astype(str).apply(nrc_dominant)
print(data_trump['nrc_emotion'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
freq = data_trump['nrc_emotion'].value_counts().sort_index()
freq.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("NRCLex Emotion Distribution — Trump Tweets")
plt.xlabel("Emotion")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. TextBlob Sentiment Analysis

TextBlob provides a simple polarity-based sentiment score (-1 to +1). We classify tweets into three categories: Positive (polarity > 0), Negative (polarity < 0), and Neutral (polarity = 0).

In [ ]:
def textblob_sentiment(text):
    """Classify text as Positive, Negative, or Neutral using TextBlob polarity."""
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    return "Neutral"

data_trump['textblob_sentiment'] = data_trump['tweet'].astype(str).apply(textblob_sentiment)
print(data_trump['textblob_sentiment'].value_counts())

In [ ]:
counts = data_trump['textblob_sentiment'].value_counts()
plt.figure(figsize=(8, 5))
plt.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c', '#95a5a6'])
plt.title("TextBlob Sentiment Distribution — Trump Tweets")
plt.xlabel("Sentiment")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 3. RoBERTa Transformer Classification

We use our fine-tuned RoBERTa model (`roberta-tweet-emotion-finetuned`) for emotion classification. This model was fine-tuned on 100K political tweets from our dataset (see `04_roberta_finetuning.ipynb`) starting from `cardiffnlp/twitter-roberta-base`, a RoBERTa variant pre-trained on ~58M tweets.

Fine-tuning on domain-specific political tweets allows the model to capture sentiment patterns unique to political discourse that generic models may miss.

Model: Fine-tuned from `cardiffnlp/twitter-roberta-base`

In [ ]:
# Load fine-tuned RoBERTa model (trained in 04_roberta_finetuning.ipynb)
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/roberta-tweet-emotion-finetuned"

classifier = pipeline(
    "text-classification",
    model=FINETUNED_MODEL_PATH,
    tokenizer=FINETUNED_MODEL_PATH,
    top_k=None
)

def roberta_dominant(text):
    """Return the dominant emotion predicted by the fine-tuned RoBERTa model."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    preds = classifier(text[:512])
    return max(preds[0], key=lambda d: d['score'])['label']

data_trump['roberta_emotion'] = data_trump['tweet'].apply(roberta_dominant)
print(data_trump['roberta_emotion'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
freq = data_trump['roberta_emotion'].value_counts()
freq.plot(kind='bar', color='coral', edgecolor='black')
plt.title("RoBERTa Emotion Distribution — Trump Tweets")
plt.xlabel("Emotion")
plt.ylabel("Number of Tweets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Summary

This analysis compares three distinct approaches to sentiment and emotion analysis:

- **NRCLex** provides lexicon-based emotion classification with broad coverage but limited context awareness
- **TextBlob** offers simple polarity-based sentiment that is fast and interpretable but lacks nuance
- **RoBERTa** leverages deep learning on Twitter-specific training data, capturing contextual semantics and achieving superior performance

The combination of these methods provides a comprehensive view of sentiment and emotion in the Trump tweet dataset, with each approach offering complementary insights.